# CoAD on Kaggle: M9 (part 14 of CoAD_reimplementation.ipynb)

Needs the saved models: the dataset must include `runs/reimpl/`. Run all cells (or Save Version > Save & Run All). Download `to_drive.zip` from the Output tab at the end.

In [ ]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Kaggle instead of Google Drive: the Drive folder CoAD_project is attached as a private dataset (read-only).
# Small folders are copied to a writable CoAD_project; the data archive and the saved models are read in place.
# Result files of earlier versions of this notebook, attached as inputs, are added back so finished runs are skipped.
KAGGLE_TAG = 'kaggle-m9'
INPUT = Path('/kaggle/input')
candidates = [p.parents[1] for p in INPUT.rglob('env/commit.txt')
              if (p.parents[1] / 'data_source' / 'TSB-AD-U.zip').is_file()]
assert candidates, 'Attach the dataset with the CoAD_project folder (env/, eda/, results/, data_source/)'
source = max(candidates, key=lambda d: (d / 'runs' / 'reimpl').is_dir())
STORE = Path('/kaggle/working/CoAD_project')
if not STORE.exists():
    for sub in ['env', 'eda', 'results']:
        shutil.copytree(source / sub, STORE / sub)
    (STORE / 'data_source').symlink_to(source / 'data_source', target_is_directory=True)
    (STORE / 'runs').mkdir()
    if (source / 'runs' / 'reimpl').is_dir():
        (STORE / 'runs' / 'reimpl').symlink_to(source / 'runs' / 'reimpl', target_is_directory=True)
    latest = {}
    for csv in INPUT.rglob('results/*.csv'):
        if csv.name not in latest or csv.stat().st_size > latest[csv.name].stat().st_size:
            latest[csv.name] = csv
    for name, csv in latest.items():
        shutil.copy2(csv, STORE / 'results' / name)
print('Dataset:', source, '| result files:', len(list((STORE / 'results').glob('*.csv'))),
      '| saved models:', (STORE / 'runs' / 'reimpl').exists())
REPO = Path('/tmp/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)

CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_reimpl.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    assert install.returncode == 0, 'Installation failed; inspect env/install_reimpl.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_reimpl.log'
print('Commit', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip(),
      '| packages OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
assert len(list(RAW.iterdir())) == 870, 'raw/ must contain exactly 870 CSVs'

os.chdir(REPO)                       # load_dataset reads the relative path dataset/TSB-AD/raw
sys.path.insert(0, str(REPO))
from toolkit.utils import load_dataset
from evaluation.evaluator import evaluate
from model.frequency import time_to_timefreq       # reference for test T2 only
from toolkit import distort as reference_distort   # reference for test T1 only
from model.patch_soft_mask import PatchFrequencyMask  # reference for test T4 only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORTS = STORE / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Data: 870 files | device:', DEVICE)

In [ ]:
FAITHFUL_TO_CODE = False   # False: follow the paper (main version); True: follow the authors' code (D1-D21)
SEEDS = [46, 47, 48]
PATCH = 8                  # P
WINDOW_MULTIPLE = 4        # T = 4 x dominant period
N_FFT = 4                  # K, STFT size
LAMBDA = 10                # weight of the reconstruction loss
LR = 0.002
EPOCHS = 300
BATCH_SIZE = 128

def use_code_details(faithful=None):
    """Per-call override of FAITHFUL_TO_CODE (None = use the global switch)."""
    return FAITHFUL_TO_CODE if faithful is None else faithful

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_series(group):
    """One TSB-AD series (min-max scaled on train) plus the window settings of the authors' code."""
    train, test, label, period = [values[0] for values in load_dataset('TSB-AD', [str(group)])]
    multiple = WINDOW_MULTIPLE if len(train) // period > WINDOW_MULTIPLE else len(train) // period
    window = (period * multiple) // PATCH * PATCH
    period = min(period, window)
    n = len(train)
    train_stride = 2 if n < 1000 else 6 if n < 10000 else 10 if n < 60000 else PATCH if n < 100000 else period
    return dict(group=group, train=train, test=test, label=label,
                period=period, window=window, train_stride=train_stride)

print('FAITHFUL_TO_CODE =', FAITHFUL_TO_CODE, '| seeds', SEEDS)

In [ ]:
manifest64 = pd.read_csv(STORE / 'eda' / 'manifest_tsbad64.csv')
reimpl_manifest_file = STORE / 'eda' / 'manifest_reimpl.csv'
if not reimpl_manifest_file.exists():
    rng = np.random.default_rng(2026)
    chosen = [236]
    for family in ['MGAB', 'SVDB', 'IOPS', 'TODS']:
        chosen.append(int(rng.choice(sorted(manifest64.loc[manifest64['family'] == family, 'group']))))
    rows = manifest64.set_index('group').loc[chosen].reset_index()
    rows['selection_reason'] = ['Pilot series of the reproduction'] + ['default_rng(2026), one per family'] * 4
    rows.to_csv(reimpl_manifest_file, index=False)
REIMPL_MANIFEST = pd.read_csv(reimpl_manifest_file)
REIMPL_GROUPS = REIMPL_MANIFEST['group'].tolist()

overview = []
for g, family in zip(REIMPL_GROUPS, REIMPL_MANIFEST['family']):
    s = load_series(g)
    overview.append({'group': g, 'family': family, 'n_train': len(s['train']), 'n_test': len(s['test']),
                     'anomaly_%': round(100 * s['label'].mean(), 2), 'period': s['period'],
                     'window': s['window'], 'train_stride': s['train_stride']})
display(pd.DataFrame(overview))

In [ ]:
DISTORTIONS = ['uniform', 'scale', 'jitter', 'mirror']

def distort(x, period, kind, faithful=None):
    """x: clean windows (B, T, 1). Returns (distorted copy, point labels (B, T))."""
    faithful = use_code_details(faithful)
    x = x.clone()
    B, T, _ = x.shape
    n = int(period * np.random.uniform(0.1, 1))           # segment length, one draw per batch
    if n == 0:
        n = 2 if kind == 'uniform' else 1
    if n >= T:
        n = T - 2
    start = torch.randint(0, T - n, (B, 1))               # segment start, one per window
    if faithful and kind == 'mirror':
        torch.randint(0, T - n, (B, 1))                   # unused draw of the authors' mirror_flip: keeps the RNG stream aligned
    rows = torch.arange(B)[:, None]

    if kind == 'scale':
        factor = np.random.randint(2, 5)
        if faithful or np.random.rand() < 0.5:            # lengthen: first n / factor points, each repeated
            n = n // factor * factor
            source = start + torch.arange(n) // factor
        else:                                             # shorten: every factor-th point of a longer span
            source = (start + torch.arange(n) * factor).clamp(max=T - 1)
    cols = start + torch.arange(n)
    segment = x[rows, cols]                               # (B, n, 1)

    if kind == 'uniform':
        if faithful:
            x[rows, cols] = x[rows, start]
        else:
            low, high = segment.min(1, keepdim=True).values, segment.max(1, keepdim=True).values
            x[rows, cols] = low + (high - low) * torch.rand(B, 1, 1, device=x.device)
    elif kind == 'scale':
        x[rows, cols] = x[rows, source]
    elif kind == 'jitter':
        if faithful:
            std = 0.05 * (x.max(1, keepdim=True).values - x.min(1, keepdim=True).values)
        else:
            std = 0.1 ** 0.5
        x[rows, cols] = segment + torch.randn(B, n, 1, device=x.device) * std
    elif kind == 'mirror':
        if faithful or np.random.rand() < 0.5:            # across the y-axis: reverse time
            x[rows, cols] = torch.flip(segment, [1])
        else:                                             # across the x-axis: mirror values inside the segment range
            x[rows, cols] = segment.min(1, keepdim=True).values + segment.max(1, keepdim=True).values - segment
    else:
        raise ValueError(f'unknown distortion {kind}')

    labels = torch.zeros(B, T, device=x.device)
    labels[rows, cols] = 1
    return x, labels

In [ ]:
def stft_features(x):
    """x: (B, T) -> (B, T, 2 * bins): real and imaginary part of every bin, one frame per time point."""
    spectrum = torch.stft(x, N_FFT, hop_length=N_FFT // 4, window=torch.hann_window(N_FFT, device=x.device),
                          normalized=True, onesided=True, return_complex=True)
    spectrum = torch.view_as_real(spectrum)            # (B, bins, frames, 2)
    spectrum = spectrum.permute(0, 2, 1, 3).flatten(2)  # (B, frames, bins * 2)
    return spectrum[:, :x.shape[1]]                     # frames = T + 1 (centred STFT); keep T

def make_patches(x, x_freq):
    """x: (B, T), x_freq: (B, T, F) -> time patches (B, N, P) and frequency patches (B, N, P * F)."""
    B, T = x.shape
    return x.reshape(B, T // PATCH, PATCH), x_freq.reshape(B, T // PATCH, -1)

In [ ]:
def head(in_size, hidden, faithful):
    """Patch classifier: linear in the paper (Eq. 2, 3, 8), two-layer MLP in the code (D4)."""
    if faithful:
        return nn.Sequential(nn.Linear(in_size, hidden), nn.SELU(), nn.Linear(hidden, 1, bias=False))
    return nn.Sequential(nn.Linear(in_size, 1))

def gru(size, faithful):
    """Bi-GRU encoder (Section 3.2): 2 layers and dropout 0.1 in the code, 3 layers in the paper (D1, D2)."""
    return nn.GRU(size, size, num_layers=2 if faithful else 3, batch_first=True, bidirectional=True,
                  dropout=0.1 if faithful else 0.0)

def init_linear(module):
    """Xavier-uniform weights and zero biases for every linear layer, as in the authors' code."""
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


class TimeFreqClassifier(nn.Module):
    """Module 3 (Eq. 1-4) and the residual classifier of module 5 (Eq. 7-8)."""

    def __init__(self, faithful):
        super().__init__()
        self.faithful = faithful
        freq_size = PATCH * 2 * (N_FFT // 2 + 1)                  # 8 points x 3 bins x (real, imaginary) = 48
        hidden = 3 * freq_size if faithful else 24                # D5
        activation = [nn.SELU()] if faithful else []              # D18
        self.proj_freq = nn.Sequential(nn.Linear(freq_size, hidden), *activation)   # W_f^p
        self.proj_time = nn.Sequential(nn.Linear(PATCH, hidden), *activation)       # W_t^p
        self.encoder = gru(hidden, faithful)                      # shared by both branches (D3)
        self.classifier_time = head(2 * hidden, hidden, faithful)       # W_t (created first, as in the authors' code,
        self.classifier_freq = head(2 * hidden, hidden, faithful)       # W_f  so a seed gives the same initial weights)
        self.classifier_residual = head(2 * hidden, hidden, faithful)   # W_r, shared by both residual branches
        init_linear(self)

    def features(self, x):
        """x: (B, T) -> encoder outputs of the frequency and of the time patches, each (B, N, 2H)."""
        time_patches, freq_patches = make_patches(x, stft_features(x))
        projected = torch.cat([self.proj_freq(freq_patches), self.proj_time(time_patches)])
        encoded, _ = self.encoder(projected)
        return encoded.chunk(2)

    def forward(self, x):
        """Eq. 1-4: logit of A = max(A_f, A_t), (B, N, 1). Max of logits = max of probabilities (sigmoid is increasing)."""
        h_freq, h_time = self.features(x)
        return torch.maximum(self.classifier_freq(h_freq), self.classifier_time(h_time))

    def residual_logit(self, x, x_rec):
        """Eq. 7-8: logit of A^r from the feature difference between the input and its reconstruction."""
        h_freq, h_time = self.features(torch.cat([x, x_rec]))
        (freq_in, freq_rec), (time_in, time_rec) = h_freq.chunk(2), h_time.chunk(2)
        return torch.maximum(self.classifier_residual(freq_in - freq_rec),
                             self.classifier_residual(time_in - time_rec))


class SoftMaskMAE(nn.Module):
    """Module 4 (Eq. 5-6): patch embeddings softly replaced by a learned mask embedding, then reconstructed."""

    def __init__(self, faithful):
        super().__init__()
        hidden = 3 * PATCH                                        # 24 in both modes
        self.proj = nn.Linear(PATCH, hidden)                      # W_m
        self.act = nn.SELU() if faithful else nn.Identity()       # D18
        self.encoder = gru(hidden, faithful)
        self.head = nn.Linear(2 * hidden, PATCH)                  # W_o
        self.mask_embedding = nn.Parameter(torch.randn(hidden))   # E_mask, one vector repeated over patches (D19)
        init_linear(self)

    def forward(self, x, prob):
        """x: (B, T), prob: patch anomaly probability (B, N, 1) -> reconstruction X_r (B, T)."""
        B, T = x.shape
        embedding = self.act(self.proj(x.reshape(B, T // PATCH, PATCH)))
        mixed = prob * self.mask_embedding + (1 - prob) * embedding    # Eq. 5
        encoded, _ = self.encoder(mixed)
        return self.head(encoded).reshape(B, T)                         # Eq. 6


class CoADReimpl(nn.Module):
    def __init__(self, faithful=None):
        super().__init__()
        self.faithful = use_code_details(faithful)
        self.classifier = TimeFreqClassifier(self.faithful)
        self.mae = SoftMaskMAE(self.faithful)

    def forward(self, x):
        """Training pass. x: (B, T) -> logits of A and A^r (B, N, 1), reconstruction X_r (B, T)."""
        logit = self.classifier(x)
        x_rec = self.mae(x, torch.sigmoid(logit))
        return logit, self.classifier.residual_logit(x, x_rec), x_rec

    @torch.no_grad()
    def predict(self, x):
        """Inference (call .eval() first). x: (B, T) -> A_c (B, N, 1) (Eq. 9) and the reconstruction (B, T)."""
        prob = torch.sigmoid(self.classifier(x))
        x_rec = self.mae(x, prob)
        prob = (prob + torch.sigmoid(self.classifier.residual_logit(x, x_rec))) / 2
        if self.faithful:                   # D10: reconstruct again, masked with A_c
            x_rec = self.mae(x, prob)
        return prob, x_rec

    def load_official(self, official):
        """Copy the weights of an authors' PatchFrequencyMask (code mode; test T4)."""
        renamed = {k.replace('detection_net.', 'classifier.').replace('reconstruction_net.', 'mae.'): v
                   for k, v in official.state_dict().items()
                   if k.startswith(('detection_net.', 'reconstruction_net.'))}
        self.load_state_dict(renamed)

In [ ]:
def make_windows(values, window, stride):
    """values (L,) -> windows (M, window) float32 and the front padding length."""
    L = len(values)
    pad = 0 if stride == 1 or (L - window) % stride == 0 else stride - (L - window) % stride
    padded = np.concatenate([np.zeros(pad), values])
    starts = np.arange(0, len(padded) - window + 1, stride)
    return np.stack([padded[s:s + window] for s in starts]).astype(np.float32), pad

def windows_to_series(window_values, length, stride, pad):
    """window_values (M, window) -> (length,): mean over the overlapping windows, padding removed."""
    total, count = np.zeros(pad + length), np.zeros(pad + length)
    for i, values in enumerate(window_values):
        total[i * stride:i * stride + len(values)] += values
        count[i * stride:i * stride + len(values)] += 1
    return total[pad:] / count[pad:]

def patch_labels(labels):
    """Point labels (B, T) -> patch labels (B, N, 1)."""
    B, T = labels.shape
    return labels.reshape(B, T // PATCH, PATCH).amax(-1, keepdim=True)

def training_batch(clean, period, faithful, code_distortion=None):
    """Clean windows (B, T) -> model input, point labels, reconstruction target.
    code_distortion (M6-C only): distortion details D12-D15 chosen apart from the batch layout D9."""
    code_distortion = faithful if code_distortion is None else code_distortion
    distorted, labels = distort(clean.unsqueeze(-1), period, random.choice(DISTORTIONS), code_distortion)
    distorted = distorted.squeeze(-1)
    if faithful:                                    # D9: clean copy + distorted copy
        return torch.cat([clean, distorted]), torch.cat([torch.zeros_like(labels), labels]), torch.cat([clean, clean])
    return distorted, labels, clean

def coad_loss(model, x, labels, target, code_loss=None):
    """Eq. 10. Returns (total, classification term, reconstruction term).
    code_loss (M6-C only): loss details D6-D8, D20 chosen apart from the model."""
    logit, residual_logit, x_rec = model(x)
    y = patch_labels(labels)
    if (model.faithful if code_loss is None else code_loss):
        pos_weight = torch.tensor([2.0], device=x.device)                                    # D6
        classify = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight) +
                    F.binary_cross_entropy_with_logits(residual_logit, y, pos_weight=pos_weight)) / 2   # D7
        error = (x_rec - target).pow(2).reshape(y.shape[0], y.shape[1], PATCH)
        anomalous = (error * y).sum() / (y.sum() * PATCH + 1e-7)
        normal = (error * (1 - y)).sum() / ((1 - y).sum() * PATCH + 1e-7)
        recon = 0.5 * anomalous + 0.5 * normal                                               # D8
    else:
        prob = (torch.sigmoid(logit) + torch.sigmoid(residual_logit)) / 2                    # A_c, Eq. 9
        classify = F.binary_cross_entropy(prob, y)
        recon = PATCH * F.mse_loss(x_rec, target)      # Eq. 10: squared norm summed over a patch, averaged over patches (D20)
    return classify + LAMBDA * recon, classify, recon

In [ ]:
def train_model(series, faithful=None, seed=46, epochs=None, log_every=50, groups=None):
    """Train CoADReimpl on the training part of one series. Returns the model and the mean losses per epoch.
    groups (M6-C only): set of code-only detail groups (section 12) applied on top of paper mode."""
    epochs = EPOCHS if epochs is None else epochs
    set_seed(seed)
    if groups is None:
        faithful = use_code_details(faithful)
        code_distortion = faithful
        model = CoADReimpl(faithful).to(DEVICE)
    else:
        faithful, code_distortion = 'training' in groups, 'distortion' in groups
        model = HybridCoAD(groups).to(DEVICE)
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99) if faithful else None   # D16
    use_amp = faithful and DEVICE.type == 'cuda'                                                            # D16
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = []
    for epoch in range(epochs):
        model.train()
        sums = np.zeros(3)
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], faithful, code_distortion)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                losses = coad_loss(model, x, labels, target, faithful)
            scaler.scale(losses[0]).backward()
            scaler.step(optimizer)
            scaler.update()
            sums += [value.item() for value in losses]
        if scheduler:
            scheduler.step()
        history.append(dict(zip(['loss', 'classify', 'recon'], sums / len(loader))))
        if log_every and (epoch + 1) % log_every == 0:
            print(f'  epoch {epoch + 1}/{epochs}: ' + ', '.join(f'{k} {v:.4f}' for k, v in history[-1].items()),
                  flush=True)
    return model, pd.DataFrame(history)

In [ ]:
@torch.no_grad()
def predict_series(model, values, window, stride):
    """Whole series -> per-point A_c and reconstruction, averaged over overlapping windows."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    probs, recs = [], []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        prob, rec = model.predict(batch.to(DEVICE))
        probs.append(prob.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        recs.append(rec.cpu())
    return (windows_to_series(torch.cat(probs).numpy(), len(values), stride, pad),
            windows_to_series(torch.cat(recs).numpy(), len(values), stride, pad))

def moving_average(score, width):
    """Centred moving average keeping the length (np.convolve 'same', as in the authors' code)."""
    return np.convolve(score, np.ones(width) / width, mode='same')

def anomaly_score(model, series):
    """Per-point anomaly score of the test series."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    if not model.faithful:
        return moving_average(test_prob + np.abs(series['test'] - test_rec), period)    # Eq. 11
    train_prob, train_rec = predict_series(model, series['train'], window, period)      # D11
    recon = np.abs(np.abs(series['test'] - test_rec) - np.abs(series['train'] - train_rec).mean())
    classify = np.abs(test_prob - train_prob.mean())
    recon, classify = moving_average(recon, period), moving_average(classify, period)
    recon[:period] = classify[:period] = 0
    return (recon + classify) / 2

METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']

def score_metrics(score, series):
    """Checks of test T7, then the repository evaluator (values in 0-1)."""
    assert len(score) == len(series['label']), 'score and label lengths differ'
    assert np.isfinite(score).all(), 'score contains NaN or Inf'
    result = evaluate(ground_truth=series['label'], anomaly_scores=score, subsequence=series['period'], mode='test')
    row = {k: float(result.point_wise[k]) for k in METRIC_KEYS[:3]}
    row.update({k: float(result.vus[k]) for k in METRIC_KEYS[3:]})
    return row

def plot_scores(series, scores, path=None):
    """Test series on top, one min-max scaled score per row below; labelled anomalies shaded."""
    fig, axes = plt.subplots(len(scores) + 1, 1, figsize=(14, 1.6 * (len(scores) + 1)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    rows = [('test data', series['test'])] + list(scores.items())
    for ax, (name, values) in zip(axes, rows):
        values = np.asarray(values, dtype=float)
        ax.plot((values - values.min()) / (values.max() - values.min() + 1e-12), lw=0.6)
        ax.fill_between(np.arange(len(values)), 0, 1, where=series['label'] == 1, color='C3', alpha=0.25,
                        transform=ax.get_xaxis_transform())
        ax.set_ylabel(name, rotation=0, ha='right', fontsize=8)
        ax.set_yticks([])
    fig.suptitle(f"group {series['group']}")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    plt.show()

In [ ]:
RESULTS_DIR = STORE / 'results'
RUNS = STORE / 'runs' / 'reimpl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

SESSION_TAG = KAGGLE_TAG            # unique per notebook copy running at the same time
RESULTS = RESULTS_DIR / f'reimpl_results_{SESSION_TAG}.csv'
RUN_MODES = ['paper', 'code']
RUN_GROUPS = REIMPL_GROUPS + [g for g in manifest64['group'] if g not in REIMPL_GROUPS]   # 5 chosen series first, then the other 59
RUN_SEEDS = SEEDS
TIME_BUDGET_MIN = 600   # Kaggle: stop well before the 12 h limit            # no new run after this many minutes; re-run 7.2 to continue
COLUMNS = ['run_id', 'mode', 'group', 'seed', 'status', 'seconds', 'error'] + METRIC_KEYS

def read_reimpl_results():
    """Rows of every session, oldest run first."""
    files = sorted(RESULTS_DIR.glob('reimpl_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.sort_values('run_id', kind='stable').reset_index(drop=True)

def run_reimpl(mode, group, seed):
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{mode}_{group}_s{seed}"
    out = RUNS / run_id
    out.mkdir(parents=True)
    row = dict(run_id=run_id, mode=mode, group=group, seed=seed)
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=(mode == 'code'), seed=seed, log_every=0)
        score = anomaly_score(model, series)
        metrics = score_metrics(score, series)
        torch.save(model.state_dict(), out / 'model.pt')
        np.save(out / 'scores.npy', score.astype(np.float32))
        history.to_csv(out / 'loss.csv', index=False)
        (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
        row.update(status='success', **metrics)
    except Exception as error:
        row.update(status='failed', error=f'{type(error).__name__}: {error}'[:300])
    row['seconds'] = round(time.time() - start, 1)
    config = dict(row, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patch=PATCH, n_fft=N_FFT, lam=LAMBDA,
                  commit=commit, device=str(DEVICE))
    (out / 'config.json').write_text(json.dumps(config, indent=2, default=str))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

print('Session', SESSION_TAG, '| writes to', RESULTS.name, '|', len(RUN_MODES) * len(RUN_GROUPS) * len(RUN_SEEDS), 'runs planned')

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon

METRICS4 = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
BASELINES = ['Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
groups64 = manifest64['group'].tolist()
family_of = manifest64.set_index('group')['family']

reproduced = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))], ignore_index=True)
reproduced = (reproduced.query("status == 'success'").sort_values('run_id')
              .drop_duplicates(['method', 'group'], keep='last'))
reimpl = (read_reimpl_results().query("status == 'success'")
          .drop_duplicates(['mode', 'group', 'seed'], keep='last'))

def metric_matrix(key):
    """64 series x methods, values x100."""
    table = reproduced.pivot(index='group', columns='method', values=key)
    for mode in ['code', 'paper']:
        table[f'CoAD reimpl {mode}'] = reimpl[reimpl['mode'] == mode].groupby('group')[key].mean()
    table = 100 * table.reindex(groups64)[['CoAD', 'CoAD reimpl code', 'CoAD reimpl paper'] + BASELINES]
    assert table.notna().all().all(), f'missing values for {key}: complete the runs first'
    return table

MATRIX = {name: metric_matrix(key) for key, name in METRICS4.items()}
print('64 series x', MATRIX['VUS-PR'].shape[1], 'methods | reimplementation seeds per mode:',
      reimpl.groupby('mode')['seed'].nunique().to_dict())
display(pd.DataFrame({name: m.mean() for name, m in MATRIX.items()}).round(2))

In [ ]:
def holm(pvalues):
    """Holm step-down adjustment."""
    pvalues = np.asarray(pvalues)
    adjusted, running = np.empty(len(pvalues)), 0.0
    for i, idx in enumerate(np.argsort(pvalues)):
        running = max(running, (len(pvalues) - i) * pvalues[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def bootstrap_ci(diff, n=10_000, seed=2026):
    """95% interval of the mean difference, resampling series with replacement."""
    rng = np.random.default_rng(seed)
    means = rng.choice(diff, size=(n, len(diff)), replace=True).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

pairwise = []
for coad in ['CoAD', 'CoAD reimpl code']:
    for name, matrix in MATRIX.items():
        rows = []
        for baseline in BASELINES:
            diff = (matrix[coad] - matrix[baseline]).to_numpy()
            low, high = bootstrap_ci(diff)
            rows.append({'CoAD': coad, 'metric': name, 'baseline': baseline,
                         'CoAD better': int((diff > 0).sum()), 'CoAD worse': int((diff < 0).sum()),
                         'mean diff': diff.mean(), 'CI low': low, 'CI high': high,
                         'p': wilcoxon(diff).pvalue})
        for row, p_holm in zip(rows, holm([r['p'] for r in rows])):
            row['p Holm'] = p_holm
            row['significant'] = p_holm < 0.05
        pairwise += rows
pairwise = pd.DataFrame(pairwise)
pairwise.to_csv(REPORTS / 'm5_pairwise_tests.csv', index=False)
display(pairwise[pairwise['metric'] == 'VUS-PR'].round(3))
print('Not significant after Holm (all metrics):')
display(pairwise[~pairwise['significant']][['CoAD', 'metric', 'baseline', 'CoAD better', 'CoAD worse',
                                            'mean diff', 'p Holm']].round(3))

In [ ]:
@torch.no_grad()
def predict_components(model, values, window, stride):
    """Per-point A (OE), A^r (residual), A_c and the final reconstruction, averaged over overlapping windows.
    Same computation as CoADReimpl.predict (section 4.1), keeping the intermediate tensors."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    parts = {k: [] for k in ['oe', 'residual', 'combined', 'reconstruction']}
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        oe = torch.sigmoid(model.classifier(x))
        first = model.mae(x, oe)
        residual = torch.sigmoid(model.classifier.residual_logit(x, first))
        combined = (oe + residual) / 2
        final = model.mae(x, combined) if model.faithful else first      # D10
        for key, value in [('oe', oe), ('residual', residual), ('combined', combined)]:
            parts[key].append(value.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        parts['reconstruction'].append(final.cpu())
    return {k: windows_to_series(torch.cat(v).numpy(), len(values), stride, pad) for k, v in parts.items()}

@torch.no_grad()
def backup_error(model, values, window, stride, n_groups=4):
    """Module A: |x - reconstruction| when the patch itself is fully masked, independently of OE."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    out = []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        B, T = x.shape
        index = torch.arange(T // PATCH, device=x.device)
        error = torch.zeros_like(x)
        for j in range(n_groups):
            masked = (index % n_groups == j).float()
            reconstruction = model.mae(x, masked.view(1, -1, 1).expand(B, -1, 1))
            error += (x - reconstruction).abs() * masked.repeat_interleave(PATCH).view(1, T)
        out.append(error.cpu())
    return windows_to_series(torch.cat(out).numpy(), len(values), stride, pad)

def processed(raw_test, raw_train, period, smooth=True, zero_start=True):
    """Authors' code-mode processing of one source: |x - train mean|, moving average, first period zeroed.
    Returns the processed test score and its train counterpart (for calibration)."""
    mean = raw_train.mean()
    out = []
    for raw in (raw_test, raw_train):
        value = np.abs(raw - mean)
        value = moving_average(value, period) if smooth else value.copy()
        if zero_start:
            value[:period] = 0
        out.append(value)
    return out

def model_signals(model, series, values=None, train_parts=None):
    """All point signals of one model on the test series (or on `values`, e.g. a slice with an injected anomaly).
    train_parts: precomputed training-part components, reused by the stress test."""
    period, window = series['period'], series['window']
    values = series['test'] if values is None else values
    test = predict_components(model, values, window, period)
    test['backup'] = backup_error(model, values, window, period)
    if train_parts is None:
        train_parts = predict_components(model, series['train'], window, period)
        train_parts['backup'] = backup_error(model, series['train'], window, period)
    zero = values is series['test']
    error_test, error_train = np.abs(values - test['reconstruction']), np.abs(series['train'] - train_parts['reconstruction'])
    r, r_train = processed(error_test, error_train, period, zero_start=zero)
    c, c_train = processed(test['combined'], train_parts['combined'], period, zero_start=zero)
    b, b_train = processed(test['backup'], train_parts['backup'], period, zero_start=zero)
    r_raw, _ = processed(error_test, error_train, period, smooth=False, zero_start=zero)
    c_raw, _ = processed(test['combined'], train_parts['combined'], period, smooth=False, zero_start=zero)
    s = {name: np.std(t) + 1e-12 for name, t in [('r', r_train), ('c', c_train), ('b', b_train)]}
    signals = {
        'full': (r + c) / 2, 'classify': c, 'reconstruct': r, 'backup': b, 'full_unsmoothed': (r_raw + c_raw) / 2,
        'oe': test['oe'], 'residual': test['residual'], 'error': error_test,
        'calibrated': (r / s['r'] + c / s['c']) / 2,
        'calibrated + backup': (r / s['r'] + c / s['c'] + b / s['b']) / 3,
    }
    return signals, train_parts

EVENT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup', 'full_unsmoothed', 'oe', 'residual', 'error']
SPLIT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup']
VARIANTS = ['calibrated', 'backup', 'calibrated + backup']
# Fixed column orders: rows are appended model by model, so every model must write the same columns in the same order
EVENT_COLUMNS = (['run_id', 'group', 'seed', 'family', 'event', 'start', 'length', 'length_over_period', 'window_share',
                  'amplitude_z'] + [f'{s}_{k}' for s in EVENT_SIGNALS
                                    for k in (['peak', 'mean', 'middle', 'edges'] if s in SPLIT_SIGNALS else ['peak', 'mean'])])
STRESS_COLUMNS = (['group', 'seed', 'family', 'type', 'length_periods', 'offset', 'position']
                  + [f'{s}_{k}' for s in SPLIT_SIGNALS + ['oe'] for k in ['peak', 'mean', 'middle', 'edges']] + ['spill'])

def segments(label):
    edges = np.diff(np.concatenate([[0], np.asarray(label).astype(int), [0]]))
    return list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))

def percentile_against(reference):
    """Fraction of the reference (normal) values strictly below each value."""
    ordered = np.sort(reference)
    return lambda v: np.searchsorted(ordered, v, side='left') / len(ordered)

def event_stats(signal, start, end, pct):
    """Peak and mean percentile over an event; middle third vs both edge thirds for events of 3 patches or more."""
    p = pct(signal[start:end])
    row = {'peak': float(pct(np.array([signal[start:end].max()]))[0]), 'mean': float(p.mean())}
    if end - start >= 3 * PATCH:
        third = (end - start) // 3
        row.update(middle=float(p[third:2 * third].mean()), edges=float(np.r_[p[:third], p[2 * third:]].mean()))
    return row

def event_rows(series, signals, base):
    """One row per labelled event. Normal reference: label 0, after the first period."""
    label, test, period = np.asarray(series['label']), series['test'], series['period']
    normal = (label == 0) & (np.arange(len(label)) >= period)
    mu, sd = test[normal].mean(), test[normal].std() + 1e-12
    pcts = {name: percentile_against(signals[name][normal]) for name in EVENT_SIGNALS}
    rows = []
    for k, (a, b) in enumerate(segments(label)):
        row = dict(base, event=k, start=int(a), length=int(b - a), length_over_period=(b - a) / period,
                   window_share=min(1.0, (b - a) / series['window']),
                   amplitude_z=float(np.abs(test[a:b] - mu).max() / sd))
        for name in EVENT_SIGNALS:
            stats = event_stats(signals[name], a, b, pcts[name])
            if name not in SPLIT_SIGNALS:
                stats = {k2: v for k2, v in stats.items() if k2 in ('peak', 'mean')}
            row.update({f'{name}_{k2}': v for k2, v in stats.items()})
        rows.append(row)
    return rows

def alarm_stats(series, score):
    """Top-k points by score, k = number of anomalous points: share that are normal, and of those the share
    within one period of an event (spill-over) rather than far from any event."""
    label, period = np.asarray(series['label']), series['period']
    valid = np.arange(len(label)) >= period
    k = int(label[valid].sum())
    top = np.where(valid)[0][np.argsort(-score[valid])[:k]]
    normal_top = top[label[top] == 0]
    near = np.zeros(len(label), bool)
    for a, b in segments(label):
        near[max(0, a - period):b + period] = True
    return {'top_k_normal_share': len(normal_top) / max(k, 1),
            'normal_alarms_near_share': float(near[normal_top].mean()) if len(normal_top) else np.nan}

In [ ]:
M9_EVENTS = RESULTS_DIR / f'm9_events_{SESSION_TAG}.csv'
M9_MODELS = RESULTS_DIR / f'm9_models_{SESSION_TAG}.csv'
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))

def read_m9(kind):
    files = sorted(RESULTS_DIR.glob(f'm9_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_m9('models')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if (int(r.group), int(r.seed)) not in done]
todo.sort(key=lambda r: r.seed)
print(len(code_runs), 'code-mode models,', len(todo), 'to analyse')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    signals, _ = model_signals(model, series)
    gap = float(np.abs(signals['full'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: recomputed final score differs from scores.npy by {gap:.1e}'
    base = dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group])
    rows = event_rows(series, signals, base)
    model_row = dict(base, n_events=len(rows), max_score_diff=gap, **alarm_stats(series, signals['full']))
    for variant in VARIANTS:
        metrics = score_metrics(signals[variant], series)
        model_row.update({f'{variant}: {k}': v for k, v in metrics.items()})
        model_row.update({f'{variant}: {k}': v for k, v in alarm_stats(series, signals[variant]).items()})
    pd.DataFrame(rows).reindex(columns=EVENT_COLUMNS).to_csv(M9_EVENTS, mode='a', header=not M9_EVENTS.exists(), index=False)
    model_row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame([model_row]).to_csv(M9_MODELS, mode='a', header=not M9_MODELS.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} events, {model_row['seconds']:.0f} s | VUS-PR "
          + ' '.join(f"{v} {100 * model_row[f'{v}: VUS_PR']:.2f}" for v in VARIANTS), flush=True)

In [ ]:
events = read_m9('events').drop_duplicates(['group', 'seed', 'event'], keep='last')
models = read_m9('models').drop_duplicates(['group', 'seed'], keep='last')
print(len(models), 'models,', len(events), 'events')
FOUND = 0.99
events['found'] = events['full_peak'] >= FOUND
events['length bin'] = pd.cut(events['length_over_period'], [0, 0.1, 0.5, 1, 2, np.inf],
                              labels=['< 0.1 period', '0.1-0.5', '0.5-1', '1-2', '> 2 periods'])

def step_lost(row):
    if row['found']:
        return 'found'
    if row['full_unsmoothed_peak'] >= FOUND:
        return 'smoothing'
    if max(row['classify_peak'], row['reconstruct_peak']) >= FOUND:
        return 'fusion'
    if max(row['oe_peak'], row['residual_peak'], row['error_peak']) >= FOUND:
        return 'processing'
    return 'all sources weak'
events['step'] = events.apply(step_lost, axis=1)

print('Share of events found (peak percentile >= 0.99), by family:')
display(events.groupby('family')['found'].agg(['mean', 'size']).round(3))
print('Share found by event length (relative to the period):')
display(events.groupby(['family', 'length bin'], observed=True)['found'].agg(['mean', 'size']).round(3).unstack(0))
print('Where missed events are lost (share of missed events):')
step_table = pd.crosstab(events.loc[~events['found'], 'family'], events.loc[~events['found'], 'step'], normalize='index')
display(step_table.round(3))
print('Normal points among the top-k scores, and share of those near an event (spill-over), mean over models:')
display(models.groupby('family')[['top_k_normal_share', 'normal_alarms_near_share']].mean().round(3))
events.to_csv(REPORTS / 'm9_events_all.csv', index=False)
step_table.to_csv(REPORTS / 'm9_step_lost.csv')

In [ ]:
from scipy.stats import spearmanr
events['oe_found'] = events['oe_peak'] >= FOUND
rescue = events.groupby('oe_found')[[f'{s}_peak' for s in ['error', 'residual', 'reconstruct', 'full', 'backup']]] \
    .agg(lambda v: (v >= FOUND).mean())
rescue.index = rescue.index.map({False: 'OE misses', True: 'OE finds'})
rescue['events'] = events.groupby('oe_found').size().values
print('Share of events found by each signal, split by whether OE finds the event:')
display(rescue.round(3))
print('Same, OE-missed events only, by family:')
display(events[~events['oe_found']].groupby('family')[['error_peak', 'reconstruct_peak', 'full_peak', 'backup_peak']]
        .agg(lambda v: (v >= FOUND).mean()).round(3))
rho = spearmanr(events['oe_mean'], events['error_mean'], nan_policy='omit')
print(f'Spearman(OE, reconstruction error) over events: rho = {rho.statistic:.2f}, p = {rho.pvalue:.1e}')
missed = events[~events['oe_found']]
diff = missed['backup_mean'] - missed['reconstruct_mean']
print(f'OE-missed events: backup minus guided reconstruction, mean percentile {diff.mean():+.3f}, '
      f'higher on {int((diff > 0).sum())}/{len(diff)}, Wilcoxon p = {wilcoxon(diff).pvalue:.1e}')

rows = []
for key, name in METRICS4.items():
    coad = 100 * code_runs.astype({'group': int, 'seed': int}).set_index(['group', 'seed'])[key]
    row = {'metric': name, 'Table 1': {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}[name],
           'CoAD (code mode)': coad.groupby(level='seed').mean().mean()}
    for variant in VARIANTS:
        v = 100 * models.set_index(['group', 'seed'])[f'{variant}: {key}']
        common = v.index.intersection(coad.index)
        per_series = (v[common] - coad[common]).groupby(level='group').mean()
        row[variant] = v[common].groupby(level='seed').mean().mean()
        row[f'{variant} - CoAD'] = per_series.mean()
        row[f'{variant} higher on'] = f'{int((per_series > 0).sum())}/{len(per_series)}'
        row[f'{variant} p (one-sided)'] = wilcoxon(per_series, alternative='greater').pvalue
    rows.append(row)
module_a = pd.DataFrame(rows).set_index('metric')
module_a.to_csv(REPORTS / 'm9_module_a.csv')
display(module_a.T.round(3))

In [ ]:
long_events = events.dropna(subset=['full_middle']) if 'full_middle' in events else events.iloc[:0]
print(len(long_events), 'events of 3 patches or more')
rows = []
for (family, length_bin), d in long_events.groupby(['family', 'length bin'], observed=True):
    if len(d) < 5:
        continue
    row = {'family': family, 'length': length_bin, 'events': len(d)}
    for s in SPLIT_SIGNALS:
        gap = d[f'{s}_middle'] - d[f'{s}_edges']
        row[f'{s}: middle - edges'] = gap.mean()
        row[f'{s}: p'] = wilcoxon(gap).pvalue if (gap != 0).any() else np.nan
    rows.append(row)
middle_edges = pd.DataFrame(rows)
middle_edges.to_csv(REPORTS / 'm9_middle_edges.csv', index=False)
display(middle_edges.round(3))
print('Spearman(length / period, full mean percentile) within family:')
for family, d in events.groupby('family'):
    r = spearmanr(d['length_over_period'], d['full_mean'])
    print(f'  {family}: rho = {r.statistic:+.2f} (p = {r.pvalue:.1e}, {len(d)} events)')

In [ ]:
M9_STRESS = RESULTS_DIR / f'm9_stress_{SESSION_TAG}.csv'
STRESS_LENGTHS = [0.25, 0.5, 1, 2, 4]
STRESS_TYPES = ['uniform', 'level']
N_POSITIONS, OFFSETS = 4, range(PATCH)

def inject(values, start, length, kind, train_sd):
    values = values.copy()
    if kind == 'uniform':
        values[start:start + length] = values[start]
    else:
        values[start:start + length] += 3 * train_sd
    return values

def stress_cases(series):
    period, window, n = series['period'], series['window'], len(series['test'])
    longest = int(max(STRESS_LENGTHS) * period)
    blocked = np.zeros(n, bool)
    for a, b in segments(series['label']):
        blocked[max(0, a - window - longest):b + window] = True
    candidates = [s for s in range(max(2 * window, period), n - longest - 2 * window) if not blocked[s]]
    if not candidates:
        return []
    rng = np.random.default_rng(2026 + series['group'])
    positions = sorted(rng.choice(candidates, size=min(N_POSITIONS, len(candidates)), replace=False))
    cases = [(kind, L, 0, p) for kind in STRESS_TYPES for L in STRESS_LENGTHS for p in positions]
    cases += [('uniform', 1, o, p) for o in OFFSETS for p in positions[:2]]
    return cases

STRESS_GROUPS = []   # per family, the first two series (manifest order) with room for the injections
for family, d in manifest64.groupby('family', sort=False):
    chosen = []
    for g in d['group']:
        if len(chosen) == 2:
            break
        if stress_cases(load_series(int(g))):
            chosen.append(int(g))
    print(f'{family}: stress series {chosen}' + ('' if chosen else ' (no normal stretch long enough)'))
    STRESS_GROUPS += chosen

done = read_m9('stress')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in STRESS_GROUPS and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'models for the stress test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    period, window = series['period'], series['window']
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    base_signals, train_parts = model_signals(model, series)
    label = np.asarray(series['label'])
    keep = np.ones(len(label), bool)
    for a, b in segments(label):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    pcts = {s: percentile_against(base_signals[s][keep]) for s in SPLIT_SIGNALS + ['oe']}
    train_sd = series['train'].std()
    rows = []
    for kind, L, offset, position in stress_cases(series):
        length = max(1, int(round(L * period)))
        start = position + offset
        a, b = start - 2 * window, start + length + 2 * window
        values = inject(series['test'], start, length, kind, train_sd)[a:b]
        signals, _ = model_signals(model, series, values=values, train_parts=train_parts)
        s0 = start - a
        row = dict(group=group, seed=seed, family=family_of[group], type=kind, length_periods=L, offset=offset,
                   position=int(position))
        for s in SPLIT_SIGNALS + ['oe']:
            stats = event_stats(signals[s], s0, s0 + length, pcts[s])
            row.update({f'{s}_{k}': v for k, v in stats.items()})
        row['spill'] = float(pcts['full'](signals['full'][s0 + length:s0 + length + period]).mean())
        rows.append(row)
    if rows:
        pd.DataFrame(rows).reindex(columns=STRESS_COLUMNS).to_csv(M9_STRESS, mode='a', header=not M9_STRESS.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} injections', flush=True)

In [ ]:
stress = read_m9('stress')
lengths = stress[stress['offset'] == 0]
cols = ['full_mean', 'full_peak', 'classify_mean', 'reconstruct_mean', 'backup_mean', 'oe_mean', 'spill']
print('Mean percentile of the injected segment, by type and length (mean over series, positions, seeds):')
by_length = lengths.groupby(['type', 'length_periods'])[cols].mean()
display(by_length.round(3))
print('Middle third minus edges (events of 3 patches or more):')
display(lengths.dropna(subset=['full_middle']).assign(
    **{f'{s}: middle - edges': lambda d, s=s: d[f'{s}_middle'] - d[f'{s}_edges'] for s in SPLIT_SIGNALS})
    .groupby(['type', 'length_periods'])[[f'{s}: middle - edges' for s in SPLIT_SIGNALS]].mean().round(3))
offsets = stress[(stress['length_periods'] == 1) & (stress['type'] == 'uniform')]
spread = offsets.groupby(['group', 'seed', 'position'])['full_mean'].agg(lambda v: v.max() - v.min())
print(f'Patch offset: range of the full-score mean percentile over 8 offsets, median {spread.median():.3f}, '
      f'90th percentile {spread.quantile(0.9):.3f}')

u = lengths[lengths['type'] == 'uniform'].groupby(['group', 'seed', 'position', 'length_periods'])['full_mean'].mean().unstack()
drop = (u[1.0] - u[4.0]).dropna()
mid = lengths[(lengths['type'] == 'uniform') & (lengths['length_periods'] >= 2)].dropna(subset=['full_middle'])
gap = mid['full_middle'] - mid['full_edges']
holds = drop.mean() >= 0.05 and gap.mean() < 0 and wilcoxon(gap).pvalue < 0.05
print(f'H2: 1 period minus 4 periods = {drop.mean():+.3f} (needs >= 0.05); middle - edges at >= 2 periods = '
      f'{gap.mean():+.3f} (p = {wilcoxon(gap).pvalue:.1e}) -> ' + ('HOLDS' if holds else 'FAILS'))
by_length.to_csv(REPORTS / 'm9_stress_by_length.csv')

In [ ]:
# Files to put back into the Drive folder CoAD_project (same sub-folders): download to_drive.zip from the Output tab.
# Only this notebook's own result files (tag kaggle-m9), so nothing written on Colab meanwhile is overwritten.
OUT = Path('/kaggle/working/to_drive')
shutil.rmtree(OUT, ignore_errors=True)
(OUT / 'results').mkdir(parents=True)
for csv in (STORE / 'results').glob(f'*{KAGGLE_TAG}*.csv'):
    shutil.copy2(csv, OUT / 'results' / csv.name)
for sub in ['reports'] + []:
    if (STORE / sub).exists():
        shutil.copytree(STORE / sub, OUT / sub)
shutil.make_archive(str(OUT), 'zip', OUT)
print('to_drive.zip:', sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*.csv')))